# LAB 3: Age Prediction — F1 Drivers

**วิชา:** Machine Learning — ใบงานที่ 3 (Regression & Classification)

**เป้าหมาย:** ทำนาย **อายุ (age)** ของนักแข่ง F1 จากสถิติต่างๆ ด้วย Regression

**หมายเหตุ:** ในสไลด์ Age Prediction ทำนายอายุจาก *ภาพใบหน้า* (ต้องใช้ CNN/Deep Learning) แต่เราปรับมาใช้ *ข้อมูลตัวเลข* แทน เพื่อให้เข้าใจหลักการง่ายๆ ก่อน — Workflow เหมือนกัน: **Feature → Regression Model → Predicted Age → Evaluation Metrics**

## 1. Import Libraries
เพิ่ม `numpy` (คำนวณ RMSE) และ metrics 3 ตัวตามสไลด์: MAE, RMSE, R²

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Data (อ่านข้อมูล)

In [ ]:
df = pd.read_csv("f1_drivers.csv")
print("ขนาดข้อมูล:", df.shape)
df.head()

## 3. Prepare Data (เตรียมข้อมูล)

- **x (Features)** = สถิติที่ใช้ทำนาย (เทียบได้กับ 'สกัดคุณลักษณะจากใบหน้า' ในสไลด์)
- **y (Target)** = age (อายุที่ต้องการทำนาย)

In [ ]:
features = ["career_wins", "career_podiums", "career_points", "pole_positions", "championships", "fastest_laps"]
x = df[features]
y = df["age"]

print("Features ที่ใช้ทำนายอายุ:", features)
x.head()

## 4. Train/Test Split (แบ่งข้อมูล)

แบ่งข้อมูลเป็น 2 ส่วน (ตามที่เรียนในหน้า Train/Test Split):
- **Train (80%)** = ใช้สอนโมเดล
- **Test (20%)** = ใช้ทดสอบกับข้อมูลที่โมเดลไม่เคยเห็น

> `random_state=42` ทำให้ผลลัพธ์เหมือนเดิมทุกครั้งที่รัน

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("ข้อมูล Train:", x_train.shape[0], "คน")
print("ข้อมูล Test:", x_test.shape[0], "คน")

## 5. Train Model (สอนโมเดล)
สอนโมเดลด้วยข้อมูล Train เท่านั้น (ห้ามใช้ Test เพื่อป้องกัน Data Leakage)

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)
print("สอนโมเดลเสร็จแล้ว ✓")

## 6. Predict (ทำนายอายุ)
ทำนายอายุของนักแข่งในชุด Test แล้วเทียบกับอายุจริง

In [ ]:
y_pred = model.predict(x_test)

# เทียบอายุจริง vs อายุที่ทำนาย
compare = pd.DataFrame({
    "อายุจริง": y_test.values,
    "อายุทำนาย": np.round(y_pred, 1)
})
compare

## 7. Evaluation Metrics (วัดผล 3 ตัวตามสไลด์)

- **MAE** = ทำนายอายุพลาดเฉลี่ยกี่ปี (ยิ่งน้อยยิ่งดี)
- **RMSE** = คล้าย MAE แต่ลงโทษการพลาดหนักๆ มากกว่า (ยิ่งน้อยยิ่งดี)
- **R² Score** = โมเดลอธิบายข้อมูลได้ดีแค่ไหน (เข้าใกล้ 1 ยิ่งดี)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2), "ปี   (ทำนายพลาดเฉลี่ย)")
print("RMSE:", round(rmse, 2), "ปี")
print("R2  :", round(r2, 3), "  (เข้าใกล้ 1 = ดี)")

## 8. Visualize (กราฟเทียบอายุจริง vs ทำนาย)

ถ้าจุดเกาะเส้นประ (เส้นทำนายสมบูรณ์แบบ) มาก = โมเดลทำนายแม่น

In [ ]:
plt.scatter(y_test, y_pred, color='blue', label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Age')
plt.ylabel('Predicted Age')
plt.title('Age Prediction: Actual vs Predicted')
plt.legend()
plt.show()

## สรุป

- Age Prediction คืองาน **Regression** (ทำนายค่าตัวเลข = อายุ)
- Workflow: Features → Train/Test Split → Regression Model → Predict → วัดผล
- วัดผลด้วย **MAE, RMSE, R²** (ตรงกับสไลด์ Age Prediction)
- **ข้อคิด:** การทำนายอายุจากสถิติแข่งอาจไม่แม่นมาก เพราะอายุไม่ได้สัมพันธ์กับผลงานโดยตรง (นักแข่งหน้าใหม่ที่เก่งก็มี นักแข่งรุ่นเก๋าที่สถิติน้อยก็มี) — ถ้า R² ออกมาต่ำ ก็เป็นเรื่องปกติและเป็นบทเรียนที่ดีว่า *ไม่ใช่ทุก Feature จะทำนายทุก Target ได้ดี*
- เวอร์ชันจริงในสไลด์ใช้ **ภาพใบหน้า + CNN** ซึ่งทำนายอายุได้แม่นกว่ามาก (ขั้นสูงขึ้นไป)